In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from dataloaders import load_skull_data
from utilitys import create_output_dir,save_loss_to_csv


In [ ]:
def mlp(in_dim, hidden=128, out_dim=1, depth=3, act=nn.Softplus):
    layers = [nn.Linear(in_dim, hidden), act()]
    for _ in range(depth - 1):
        layers += [nn.Linear(hidden, hidden), act()]
    layers += [nn.Linear(hidden, out_dim)]
    return nn.Sequential(*layers)


class FourierFeatures1D(nn.Module):
    def __init__(self,
                 num_scales: int = 4,
                 num_freq_per_scale: int = 256,
                 base_sigma: float = 10.0,
                 gamma: float = 0.5,
                 include_input: bool = True):
        super().__init__()
        self.include_input = include_input
        self.num_scales = num_scales
        self.num_freq_per_scale = num_freq_per_scale

        sigmas = [base_sigma * (gamma ** s) for s in range(num_scales)]
        Bs = []
        for sigma in sigmas:
            B_s = torch.randn(1, num_freq_per_scale) * sigma
            Bs.append(B_s)
        B = torch.cat(Bs, dim=1)  # [1, K_tot]
        self.register_buffer("B", B)

        self.out_dim = (1 if include_input else 0) + 2 * B.shape[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # proj: [N, K_tot]
        proj = x @ self.B
        if self.include_input:
            return torch.cat([x, torch.sin(proj), torch.cos(proj)], dim=-1)
        else:
            return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


class NeuralSVD_MLP(nn.Module):

    def __init__(self,
                 input_size,          # (H, W)
                 k,
                 hidden=128,
                 depth=3,
                 use_fourier: bool = True,
                 ff_num_scales: int = 4,
                 ff_num_freq_per_scale: int = 256,
                 ff_base_sigma: float = 10.0,
                 ff_gamma: float = 0.5,
                 ff_include_input: bool = True,
                 # Sequential Nesting
                 use_sequential_nesting: bool = True):
        super().__init__()
        self.H, self.W = input_size
        self.k = k
        self.use_fourier = use_fourier
        self.use_sequential_nesting = use_sequential_nesting

        # normalized coordinates ([-1,1])
        i = torch.linspace(-1.0, 1.0, self.H).view(self.H, 1)  # [H,1]
        j = torch.linspace(-1.0, 1.0, self.W).view(self.W, 1)  # [W,1]
        self.register_buffer("row_coords", i)
        self.register_buffer("col_coords", j)

        # Fourier feature encoders
        if use_fourier:
            self.ff_row = FourierFeatures1D(
                num_scales=ff_num_scales,
                num_freq_per_scale=ff_num_freq_per_scale,
                base_sigma=ff_base_sigma,
                gamma=ff_gamma,
                include_input=ff_include_input,
            )
            self.ff_col = FourierFeatures1D(
                num_scales=ff_num_scales,
                num_freq_per_scale=ff_num_freq_per_scale,
                base_sigma=ff_base_sigma,
                gamma=ff_gamma,
                include_input=ff_include_input,
            )
            in_dim = self.ff_row.out_dim  # = self.ff_col.out_dim
        else:
            self.ff_row = None
            self.ff_col = None
            in_dim = 1

        self.row_heads = nn.ModuleList([mlp(in_dim, hidden, 1, depth) for _ in range(k)])
        self.col_heads = nn.ModuleList([mlp(in_dim, hidden, 1, depth) for _ in range(k)])

    def forward(self, x, k=None):
        B = x.shape[0]
        r_req = self.k if k is None else int(k)
        r = max(0, min(self.k, r_req))

        r_in = self.row_coords
        c_in = self.col_coords
        if self.use_fourier:
            r_in = self.ff_row(r_in)    # [H, Din]
            c_in = self.ff_col(c_in)    # [W, Din]

        U_cols, V_cols = [], []
        for ell in range(r):
            u_col = self.row_heads[ell](r_in)   # [H,1]
            v_col = self.col_heads[ell](c_in)   # [W,1]
            U_cols.append(u_col)
            V_cols.append(v_col)

        if r == 0:
            U = x.new_zeros(B, self.H, 0)
            V = x.new_zeros(B, self.W, 0)
            X_rec = x.new_zeros(B, self.H, self.W)
            return {"U": U, "V": V, "X_rec": X_rec}

        U = torch.cat(U_cols, dim=1).unsqueeze(0).expand(B, -1, -1)  # [B,H,r]
        V = torch.cat(V_cols, dim=1).unsqueeze(0).expand(B, -1, -1)  # [B,W,r]
        X_rec = torch.bmm(U, V.transpose(1, 2))                      # [B,H,W]
        return {"U": U, "V": V, "X_rec": X_rec}

    def _compute_sequential_nesting_loss(self, x):
        """
        Sequential Nesting loss using MSE
        """
        B, H, W = x.shape
        k = self.k

        if k == 0:
            return torch.zeros((), device=x.device, requires_grad=True)

        # Get U and V columns
        r_in = self.row_coords
        c_in = self.col_coords
        if self.use_fourier:
            r_in = self.ff_row(r_in)
            c_in = self.ff_col(c_in)

        U_cols, V_cols = [], []
        for ell in range(k):
            u_col = self.row_heads[ell](r_in)   # [H,1]
            v_col = self.col_heads[ell](c_in)   # [W,1]
            U_cols.append(u_col)
            V_cols.append(v_col)

        total_loss = 0.0
        for ell in range(1, k + 1):
            U_1_to_ell = torch.cat(U_cols[:ell], dim=1).unsqueeze(0).expand(B, -1, -1)  # [B,H,ell]
            V_1_to_ell = torch.cat(V_cols[:ell], dim=1).unsqueeze(0).expand(B, -1, -1)  # [B,W,ell]

            X_rec = torch.bmm(U_1_to_ell, V_1_to_ell.transpose(1, 2))  # [B,H,W]
            mse_loss = F.mse_loss(X_rec, x)

            total_loss = total_loss + mse_loss

        return total_loss

    def loss(self, x, output):
        """
        Loss function (MSE-based)
        """
        B, H, W = x.shape
        k = output["U"].shape[-1]  # number of active components

        if k == 0:
            mse = F.mse_loss(output["X_rec"], x)
            zero = torch.zeros((), device=x.device)
            return mse, zero, mse   # return mse for all

        # MSE for monitoring
        mse = F.mse_loss(output["X_rec"], x)
        zero = torch.zeros((), device=x.device)

        if self.use_sequential_nesting:
            sequential_loss = self._compute_sequential_nesting_loss(x)
            return mse, zero, sequential_loss
        else:
            standard_loss = F.mse_loss(output["X_rec"], x)
            return mse, zero, standard_loss


In [ ]:
def train(model, data, optimizer, epochs, save_path, use_k_loop=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device)
    model.to(device).train()
    data = data.to(device)
    
    
    if use_k_loop:
        k_losses = {k: {'mse': [], 'ortho': [], 'total': []} for k in range(1, model.k + 1)}
        
        for k in range(1, model.k + 1):
            print(f"\nStart to train k={k}")
            for epoch in range(epochs):
                optimizer.zero_grad()
                output = model.forward(data, k)
                loss_recon, loss_ortho, total_loss = model.loss(data, output)
                
                # 记录损失
                k_losses[k]['mse'].append(loss_recon.item())
                k_losses[k]['ortho'].append(loss_ortho.item())
                k_losses[k]['total'].append(total_loss.item())
                
                total_loss.backward()
                optimizer.step()
                
                if epoch % 10 == 0:
                    print(f'Epoch: {epoch:3d}/{epochs}, '
                          f'MSE_loss: {loss_recon.item():.6f}, Ortho_loss {loss_ortho.item():.6f}, '
                          f'Total_loss: {total_loss.item():.6f}')
        
        save_loss_to_csv(k_losses, save_path)
        print("Everything is done! Saved to ", save_path)
        return model, k_losses
    
    else:
        losses = {'mse': [], 'ortho': [], 'total': []}
        
        print(f"\nStart to train for {epochs} epochs")
        for epoch in range(epochs):
            optimizer.zero_grad()
            output = model.forward(data)
            loss_recon, loss_ortho, total_loss = model.loss(data, output)
            losses['mse'].append(loss_recon.item())
            losses['ortho'].append(loss_ortho.item())
            losses['total'].append(total_loss.item())
            
            total_loss.backward()
            optimizer.step()
            
            if epoch % 100 == 0:
                print(f'Epoch: {epoch:3d}/{epochs}, '
                      f'MSE Loss: {loss_recon.item():.6f}, Orthogonality Loss: {loss_ortho.item():.6f}, '
                      f'Total Loss: {total_loss.item():.6f}')
        
        if hasattr(model, 'info'):
            with open(os.path.join(save_path, 'model_info.txt'), 'w', encoding='utf-8') as f:
                f.write(model.info())
        
        save_loss_to_csv(losses, save_path)
        print("All operations completed! Save path:", save_path)
        return model, losses

In [ ]:
dataH, input_sizeH, data_nameH = load_skull_data(axis="H")
dataW, input_sizeW, data_nameW = load_skull_data(axis="W")
dataD, input_sizeD, data_nameD = load_skull_data(axis="D")
data,input_size,_ = load_skull_data()
print(input_sizeH,input_sizeW,input_sizeD,input_size)

In [ ]:
k=8
modelH = NeuralSVD_MLP(k=k, input_size=input_sizeH)
optimizerH = torch.optim.Adam(modelH.parameters(), lr=0.001)

modelW = NeuralSVD_MLP(k=k, input_size=input_sizeW)
optimizerW = torch.optim.Adam(modelW.parameters(), lr=0.001)

modelD = NeuralSVD_MLP(k=k, input_size=input_sizeD)
optimizerD = torch.optim.Adam(modelD.parameters(), lr=0.001)



In [ ]:
save_path, _ = create_output_dir("HOSVD", data_nameH, k)
save_pathH = os.path.join(save_path, "H")  
save_pathW = os.path.join(save_path, "W")  
save_pathD = os.path.join(save_path, "D")   


In [ ]:
epoch = 4000

modelH,_ = train(modelH, dataH, optimizerH, epoch ,save_pathH,use_k_loop=1)
modelW,_ = train(modelW, dataW, optimizerW, epoch ,save_pathW,use_k_loop=1)
modelD,_ = train(modelD, dataD, optimizerD, epoch ,save_pathD,use_k_loop=1)


In [ ]:
def left_pinv(U: torch.Tensor, rcond: float = 1e-4) -> torch.Tensor:
    return torch.linalg.pinv(U, rcond=rcond)

def projection_core(X_like: torch.Tensor,
                    Uh: torch.Tensor, Uw: torch.Tensor, Ud: torch.Tensor,
                    rcond: float = 1e-4,
                    detach: bool = True,
                    kind: str | None = None) -> torch.Tensor:
    """
    Direct projection in BDHW order:
      S* = X ×_D Ud† ×_H Uh† ×_W Uw†
    Args:
        X_like: [B, D, H, W] (if passed as 3D unfolded, no reordering is performed here)
        Uh: [H, r], Uw: [W, r], Ud: [D, r]
    Returns:
        S: [B, r, r, r] (order: r_D, r_H, r_W -> 'bzxy')
    """
    # No reordering, directly assume X_like is [B, D, H, W]
    X_bhwd = X_like
    dev = X_bhwd.device
    Uh, Uw, Ud = Uh.to(dev), Uw.to(dev), Ud.to(dev)

    # Left pseudoinverse
    Ph = left_pinv(Uh, rcond=rcond)   # [r, H] -> denote as xh
    Pw = left_pinv(Uw, rcond=rcond)   # [r, W] -> denote as yw
    Pd = left_pinv(Ud, rcond=rcond)   # [r, D] -> denote as zd

    S = torch.einsum('bhwd, xh, yw, zd -> b x y z', X_bhwd, Ph, Pw, Pd)
    return S.detach() if detach else S

def rotate_for_view_hwd(x: torch.Tensor, pitch_k: int = 1, yaw_k: int = 1) -> torch.Tensor:
    # Bottom-to-top: rotate in (H, D) plane
    x = torch.rot90(x, k=pitch_k % 4, dims=(3, 1))
    # Left-to-right: rotate in (W, D) plane
    x = torch.rot90(x, k=yaw_k % 4, dims=(2, 3))
    return x

def reconstruct_from_core(S: torch.Tensor,
                          Uh: torch.Tensor, Uw: torch.Tensor, Ud: torch.Tensor) -> torch.Tensor:
    dev = S.device
    Uh, Uw, Ud = Uh.to(dev), Uw.to(dev), Ud.to(dev)

    # 'bzxy, dz, hx, wy -> bdhw'
    X_rec = torch.einsum('bxyz, hx, wy, dz -> bhwd', S, Uh, Uw, Ud)
    return X_rec


In [ ]:
from visualization import visualize_3d_results
def visualize_wrapper_3d(data, model_rec, k, save_path):
    fig = visualize_3d_results(
        original_data=data,
        model_rec=model_rec,
        k=k,
        save_path=save_path
    )
    return fig


In [ ]:
import torch.nn.functional as F

Uh,Uw,Ud = modelH(dataH)["U"][0],modelW(dataW)["U"][0],modelD(dataD)["U"][0]

S = projection_core(data, Uh, Uw, Ud, rcond=1e-4, detach=True)  # [B,r,r,r]
X_rec = reconstruct_from_core(S, Uh, Uw, Ud)            
mse = F.mse_loss(X_rec, data)
print("proj-core MSE:", mse.item())


X_rot = rotate_for_view_hwd(X_rec)
data = rotate_for_view_hwd(data)


In [ ]:
visualize_wrapper_3d(data,X_rot,k,save_path)

In [ ]:
import matplotlib.pyplot as plt
import os

def visualize_results(U1,U2,U3, path):
    plt.figure(figsize=(15, 5))
   
       
    U1_norm = U1 / torch.norm(U1, dim=0, keepdim=True)
    U1_ortho = torch.matmul(U1_norm.t(), U1_norm)
    plt.subplot(131)
    plt.imshow(U1_ortho.cpu().detach().numpy(), cmap='viridis_r', vmin=-1, vmax=1)
    plt.colorbar()
    
    U2_norm = U2 / torch.norm(U2, dim=0, keepdim=True)
    U2_ortho = torch.matmul(U2_norm.t(), U2_norm)
    plt.subplot(132)
    plt.imshow(U2_ortho.cpu().detach().numpy(), cmap='viridis_r', vmin=-1, vmax=1)
    plt.colorbar()
    
    U3_norm = U3 / torch.norm(U3, dim=0, keepdim=True)
    U3_ortho = torch.matmul(U3_norm.t(), U3_norm)
    plt.subplot(133)
    plt.imshow(U3_ortho.cpu().detach().numpy(), cmap='viridis_r', vmin=-1, vmax=1)
    plt.colorbar()
    
    plt.tight_layout()
   
    os.makedirs(path, exist_ok=True)
    plt.savefig(os.path.join(path, 'orthogonality_comparison.png'))
    plt.close()

In [ ]:
visualize_results(Ud,Uw,Uh,save_path)